# DeepFP_Prep — 14 `allowed` embeddings × 3 random SMILES

This notebook mirrors a **post-clone setup with checkpoints under `DeepFP_Prep/utils/assets`**, using the same `feature_process.py` / `embed.py` as the repository.

**Environment:** use the conda env from `cmd_fp.yml` (on some machines it may still be named `cmd_smi`). Requires RDKit, PyTorch, transformers, torch_geometric, OGB, Uni-Mol, etc.

**Embedding set:** intersection of `feature_process.ALLOWED_EMBEDDINGS` and `Embedding.available()` — **14** names (ChemBERTa is not in this subset).

In [1]:
from pathlib import Path
import os, sys

def _find_repo() -> Path:
    candidates = [
        Path("/home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main"),
        Path("/home/zyrlia/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main"),
    ]
    for p in candidates:
        if (p / "DeepFP_Prep" / "feature_process.py").is_file():
            return p.resolve()
    raise FileNotFoundError(
        "ArcMol-main repo not found; set REPO_ROOT to your clone path."
    )

REPO_ROOT = _find_repo()
DEEPFP_DIR = REPO_ROOT / "DeepFP_Prep"
sys.path.insert(0, str(DEEPFP_DIR))
os.chdir(DEEPFP_DIR)
print("REPO_ROOT =", REPO_ROOT)
print("ASSETS    =", DEEPFP_DIR / "utils" / "assets")

REPO_ROOT = /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main
ASSETS    = /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/DeepFP_Prep/utils/assets


In [2]:
import numpy as np
import pandas as pd
from feature_process import ALLOWED_EMBEDDINGS, resolve_embedding_names, csv_to_pkls

names = resolve_embedding_names("allowed")
assert len(names) == len(ALLOWED_EMBEDDINGS) == 14, (len(names), len(ALLOWED_EMBEDDINGS))
print("ALLOWED_EMBEDDINGS (14):", sorted(ALLOWED_EMBEDDINGS))
print("Resolved this session:", names)

ALLOWED_EMBEDDINGS (14): ['AttrMask', 'BioT5', 'EStateFingerprint', 'GPT-GNN', 'GROVER', 'GraphCL', 'GraphMVP', 'MACCSkeys', 'MolCLR', 'MolT5', 'RDKFingerprint', 'UniMolV1', 'UniMolV2_1.1B', 'UniMolV2_84M']
Resolved this session: ['AttrMask', 'BioT5', 'EStateFingerprint', 'GPT-GNN', 'GROVER', 'GraphCL', 'GraphMVP', 'MACCSkeys', 'MolCLR', 'MolT5', 'RDKFingerprint', 'UniMolV1', 'UniMolV2_1.1B', 'UniMolV2_84M']


In [3]:
rng = np.random.default_rng(42)
POOL = [
    "CCO", "c1ccccc1", "CC(=O)O", "CCN(CC)CC",
    "Cc1ccccc1O", "C1CCCCC1", "CC(C)CC(C)C", "CC(C)=O",
]
idx = rng.choice(len(POOL), size=3, replace=False)
SMILES_demo = [POOL[int(i)] for i in sorted(idx)]
print("Random 3 SMILES (seed=42):", SMILES_demo)

NB_DIR = REPO_ROOT / "notebook"
NB_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = NB_DIR / "_demo_smiles_input.csv"
OUT_DIR = NB_DIR / "_demo_pkls_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"smiles": SMILES_demo}).to_csv(CSV_PATH, index=False)
print("Wrote", CSV_PATH)

Random 3 SMILES (seed=42): ['CCO', 'Cc1ccccc1O', 'CC(C)CC(C)C']
Wrote /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/notebook/_demo_smiles_input.csv


In [4]:
out_files = csv_to_pkls(
    str(CSV_PATH),
    str(OUT_DIR),
    smiles_col="smiles",
    y_col="y",
    split_col="split",
    embedding_names=names,
    batch_size=8,
    chunk_size=10000,
    include_rdkit=True,
)
print("\nOutput files:\n", out_files)

[WARN] Target column 'y' not found; writing features only (no target).
[OK] all: rows 0..2 -> /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/notebook/_demo_pkls_output/all_batch_0_3.pkl

Output files:
 ['/home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/notebook/_demo_pkls_output/all_batch_0_3.pkl']


In [5]:
import pickle

pkl_files = sorted(OUT_DIR.glob("all_batch_*.pkl"))
assert pkl_files, "No output PKL found; check the previous cell for errors."
pkl_path = pkl_files[0]
print("PKL:", pkl_path)

with open(pkl_path, "rb") as f:
    pack = pickle.load(f)

rows = []
for rid, rec in sorted(pack.items()):
    row = {"row_id": rid, "SMILES": rec["SMILES"]}
    rdk = rec.get("rdkit_descriptors")
    row["n_rdkit_descriptors"] = len(rdk) if isinstance(rdk, dict) else 0
    for k in names:
        v = rec[k]
        a = np.asarray(v)
        row[f"{k}_dim"] = int(a.shape[-1]) if a.ndim else int(a.size)
    rows.append(row)

summary = pd.DataFrame(rows)
show_cols = ["row_id", "SMILES", "n_rdkit_descriptors"] + [f"{names[0]}_dim", f"{names[1]}_dim", f"{names[2]}_dim"]
print("\nPer-molecule summary (first columns):")
print(summary[show_cols].to_string(index=False))

dim_cols = [c for c in summary.columns if c.endswith("_dim")]
print("\nEmbedding vector dimensions (row 0):")
print(summary[dim_cols].iloc[0].to_string())

print("\n" + "=" * 70)
print(
    "[OK] Pipeline finished: 3 molecules × 14 allowed embeddings + RDKit descriptors → PKL."
)
print("=" * 70)

PKL: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/notebook/_demo_pkls_output/all_batch_0_3.pkl

Per-molecule summary (first columns):
   row_id          SMILES  n_rdkit_descriptors  AttrMask_dim  BioT5_dim  EStateFingerprint_dim
0       0             CCO                  217           300        768                     79
1       1     Cc1ccccc1O                  217           300        768                     79
2       2  CC(C)CC(C)C                  217           300        768                     79

Embedding vector dimensions (row 0):
AttrMask_dim             300
BioT5_dim                768
EStateFingerprint_dim     79
GPT-GNN_dim             2308
GROVER_dim              3200
GraphCL_dim             2308
GraphMVP_dim             300
MACCSkeys_dim            167
MolCLR_dim               300
MolT5_dim                768
RDKFingerprint_dim        2048
UniMolV1_dim             512
UniMolV2_1.1B_dim       1280
UniMolV2_84M_dim         512

[OK] Pipeline finished: 3 molecu